# Ch 7 · Exercise 02 — Reading the Spark UI: skew, shuffle, and spill  [local Spark]

Runs on a local Apache Spark install — `pip install pyspark`, with Java 17 — not on Databricks. The Spark UI
it produces is the same one a Databricks classic cluster shows.

The notebook joins a large table to a small one, first with evenly spread keys, then with **80% of rows on a
single key**. It runs the skewed join with adaptive query execution **off**, then **on**, and finishes with a
sort starved of memory so that it **spills**. After each run it reads the Spark UI's own REST interface and
prints the numbers the Stages tab shows: task time and shuffle read, at minimum, median, and maximum.

**Before running:** start from a fresh kernel. `DRIVER_MEMORY` only takes effect when Spark's JVM first starts.

In [1]:
# Settings. DRIVER_MEMORY takes effect only in a fresh kernel.
N_ROWS = 10_000_000          # rows in the large table; use 5_000_000 if runs feel slow
DRIVER_MEMORY = "1g"         # deliberately small, so the sort and the skewed join spill
SHUFFLE_PARTITIONS = 20

from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder
         .master("local[*]")
         .appName("spark-ui-skew-spill")
         .config("spark.driver.memory", DRIVER_MEMORY)
         .config("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
         .config("spark.sql.autoBroadcastJoinThreshold", "-1")      # force shuffle joins
         .config("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "16m")
         .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "16m")
         .getOrCreate())
sc = spark.sparkContext
heap_mb = sc._jvm.java.lang.Runtime.getRuntime().maxMemory() // (1024 * 1024)
print("Spark", spark.version, "| cores", sc.defaultParallelism, "| JVM heap", heap_mb, "MB")
print("Spark UI:", sc.uiWebUrl, "| application:", sc.applicationId)
if heap_mb > 2048:
    print("NOTE: the heap is larger than DRIVER_MEMORY asks for, so this kernel's JVM started earlier.",
          "Restart the kernel to apply it.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/15 06:43:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.4 | cores 10 | JVM heap 1024 MB
Spark UI: http://localhost:4040 | application: local-1792061009112


## A helper that reads the Spark UI

Every run is labeled. Afterwards, `report` asks the Spark UI's REST interface for that run's stages, picks the
one that read the most shuffle data, and prints its task summary — the same figures as the **Summary Metrics**
table at the top of that stage's page in the **Stages** tab.

In [2]:
import json, urllib.request
API = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"

def _get(path):
    with urllib.request.urlopen(API + path) as r:
        return json.load(r)

def mb(b):
    return f"{b / 1024 / 1024:,.1f} MB"

def run(label, df, adaptive):
    """Execute df completely, writing nowhere, then report on it."""
    spark.conf.set("spark.sql.adaptive.enabled", "true" if adaptive else "false")
    sc.setJobDescription(label)
    df.write.format("noop").mode("overwrite").save()
    sc.setJobDescription(None)
    report(label)

def report(label):
    jobs = [j for j in _get("/jobs") if j.get("description") == label]
    stages = [st for sid in sorted({s for j in jobs for s in j["stageIds"]})
                 for st in _get(f"/stages/{sid}") if st["status"] == "COMPLETE"]
    if not stages:
        print(f"\n=== {label}\n  no completed stages found")
        return
    st = max(stages, key=lambda s: (s.get("shuffleReadBytes", 0), s["executorRunTime"]))
    q = _get(f"/stages/{st['stageId']}/{st['attemptId']}/taskSummary?quantiles=0.0,0.5,1.0")
    rt, rb = q["executorRunTime"], q["shuffleReadMetrics"]["readBytes"]
    print(f"\n=== {label}")
    print(f"  stages: {len(stages)}   heaviest stage: {st['stageId']}   tasks in it: {st['numTasks']}")
    print(f"  task time     min {rt[0]/1000:7.1f} s  median {rt[1]/1000:7.1f} s  max {rt[2]/1000:7.1f} s")
    print(f"  shuffle read  min {mb(rb[0]):>11}  median {mb(rb[1]):>11}  max {mb(rb[2]):>11}")
    print(f"  spill, all stages:  memory {mb(sum(s.get('memoryBytesSpilled', 0) for s in stages))}"
          f"   disk {mb(sum(s.get('diskBytesSpilled', 0) for s in stages))}")

## The data

A large table with a 52-character payload per row, in two versions: **uniform**, with keys spread evenly over
1,000 values, and **skewed**, with 80% of rows on key `0`. The small table has one row per key.

In [3]:
# Ten input partitions, set explicitly: with a single partition, Spark can skip the shuffle entirely,
# and the exercise would then depend on how many cores the machine has.
base = spark.range(0, N_ROWS, 1, numPartitions=10).withColumn(
    "payload", F.concat(F.lpad(F.col("id").cast("string"), 12, "0"), F.repeat(F.lit("x"), 40)))
uniform = base.withColumn("key", F.col("id") % 1000)
skewed  = base.withColumn("key", F.when(F.col("id") % 10 < 8, F.lit(0)).otherwise(F.col("id") % 1000))
dim = (spark.range(0, 1000, 1, numPartitions=2).withColumnRenamed("id", "key")
       .withColumn("label", F.concat(F.lit("k"), F.col("key").cast("string"))))
print("rows:", N_ROWS)

rows: 10000000


## 1. Baseline — uniform keys, adaptive query execution off

In [4]:
run("1 · uniform keys, adaptive off", uniform.join(dim, "key"), adaptive=False)


=== 1 · uniform keys, adaptive off
  stages: 3   heaviest stage: 2   tasks in it: 20
  task time     min     0.2 s  median     0.8 s  max     1.1 s
  shuffle read  min      4.5 MB  median      6.0 MB  max      7.9 MB
  spill, all stages:  memory 670.0 MB   disk 73.4 MB


## 2. Skew — 80% of rows on one key, adaptive query execution off

One shuffle partition receives key `0`, so one task does most of the work. Compare **max** with **median**.

In [5]:
run("2 · skewed keys, adaptive off", skewed.join(dim, "key"), adaptive=False)

[Stage 5:======================================================>  (19 + 1) / 20]


=== 2 · skewed keys, adaptive off
  stages: 3   heaviest stage: 5   tasks in it: 20
  task time     min     0.0 s  median     0.1 s  max     1.8 s
  shuffle read  min      0.7 MB  median      1.3 MB  max     73.6 MB
  spill, all stages:  memory 636.0 MB   disk 64.7 MB


## 3. The same skewed join, adaptive query execution on

Its skew-join handling detects the oversized partition and splits it across several tasks.

In [6]:
run("3 · skewed keys, adaptive on", skewed.join(dim, "key"), adaptive=True)


=== 3 · skewed keys, adaptive on
  stages: 3   heaviest stage: 10   tasks in it: 8
  task time     min     0.2 s  median     0.5 s  max     0.6 s
  shuffle read  min      5.8 MB  median     14.7 MB  max     14.7 MB
  spill, all stages:  memory 714.0 MB   disk 74.0 MB


## 4. Spill — a sort with too little memory

Everything goes into two partitions, so each of two tasks must sort half the table. With a small heap, the
sort cannot fit in memory and spills to disk.

In [7]:
run("4 · sort into two partitions", uniform.repartition(2).sortWithinPartitions("payload"), adaptive=False)

[Stage 12:>                                                         (0 + 2) / 2]


=== 4 · sort into two partitions
  stages: 2   heaviest stage: 12   tasks in it: 2
  task time     min     2.8 s  median     2.8 s  max     2.8 s
  shuffle read  min     87.3 MB  median     87.3 MB  max     87.3 MB
  spill, all stages:  memory 1,384.0 MB   disk 273.8 MB


## 5. Coalescing — 200 shuffle partitions for a small result

An aggregation down to 1,000 rows, with the default-sized 200 shuffle partitions. Adaptive query execution
merges the many near-empty partitions; compare the number of tasks.

In [8]:
spark.conf.set("spark.sql.shuffle.partitions", 200)
counts = uniform.groupBy("key").count()
run("5a · 200 partitions, adaptive off", counts, adaptive=False)
run("5b · 200 partitions, adaptive on",  counts, adaptive=True)
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)


=== 5a · 200 partitions, adaptive off
  stages: 2   heaviest stage: 14   tasks in it: 200
  task time     min     0.0 s  median     0.0 s  max     0.0 s
  shuffle read  min      0.0 MB  median      0.0 MB  max      0.0 MB
  spill, all stages:  memory 0.0 MB   disk 0.0 MB

=== 5b · 200 partitions, adaptive on
  stages: 2   heaviest stage: 17   tasks in it: 1
  task time     min     0.0 s  median     0.0 s  max     0.0 s
  shuffle read  min      0.2 MB  median      0.2 MB  max      0.2 MB
  spill, all stages:  memory 0.0 MB   disk 0.0 MB


## What to check — in the Spark UI as well as above

Open the Spark UI address printed by the first cell. The session stays alive until you stop it.

- **Stages** — open the heaviest stage of runs 1, 2, and 3. In **Summary Metrics**, compare *Max* with
  *Median* for **Duration** and **Shuffle Read Size**. In run 2, one task stands far above the rest.
- **SQL / DataFrame** — open the query for run 3. The join reads **`SortMergeJoin(skew=true)`**, and an
  **`AQEShuffleRead`** node reports **number of skewed partitions** and **number of skewed partition splits** —
  how many pieces the oversized partition was cut into. Compare the same query for run 2, which has neither.
- **Stages**, for runs 2 and 4 — the **Spill (Memory)** and **Spill (Disk)** columns appear only when a stage
  spilled. *Spill (Memory)* is the data's size in memory before it was written out; *Spill (Disk)* is its
  serialized, compressed size on disk, which is why it is much smaller.
- **Run 5** — the number of tasks with adaptive query execution off, and with it on.

In [9]:
# When you are finished exploring the Spark UI:
# spark.stop()

In [10]:
found = False
for q in _get("/sql?details=true"):
    names = [n["nodeName"] for n in q["nodes"]]
    if any("skew=true" in name for name in names):
        found = True
        print(f"query {q['id']}: {q.get('description', '')}")
        for n in q["nodes"]:
            if "AQEShuffleRead" in n["nodeName"] or "SortMergeJoin" in n["nodeName"]:
                print("  ", n["nodeName"])
                for m in n.get("metrics", []):
                    if "skew" in m["name"] or "coalesced" in m["name"]:
                        print("      ", m["name"], "=", m["value"])
if not found:
    print("no query with skew handling found — is the session from the run still alive?")

query 2: 3 · skewed keys, adaptive on
   AQEShuffleRead
       number of coalesced partitions = 3
   AQEShuffleRead
       number of coalesced partitions = 3
       number of skewed partitions = 1
       number of skewed partition splits = 5
   SortMergeJoin(skew=true)
